# <u> <strong> Textual Analysis using Machine Learning </strong> </u>

Nowadays, machine learning is used to analyse vast quantities of textual data, saving considerable time that might otherwise be spent manually analyzing text. In this notebook, consumer complaints to banks are analyzed for greater insights.

In [18]:
import re
from nltk.stem import WordNetLemmatizer, PorterStemmer, SnowballStemmer
import pandas as pd

In [19]:
text_data = pd.read_csv(r"C:\Users\Lenovo\Downloads\main_consumer_complaints.csv")
text_data.head()

,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company,state,tags,submitted_via
0,Debt collection,"Other (i.e. phone, health club, etc.)",Cont'd attempts collect debt not owed,Debt was paid,XXXX has claimed I owe them {$27.00} for XXXX ...,"Diversified Consultants, Inc.",NY,Older American,Web
1,Consumer Loan,Vehicle loan,Managing the loan or lease,NaN,Due to inconsistencies in the amount owed that...,M&T Bank Corporation,VA,Servicemember,Web
2,Mortgage,Conventional fixed mortgage,"Loan modification,collection,foreclosure",NaN,In XX/XX/XXXX my wages that I earned at my job...,Wells Fargo & Company,CA,NaN,Web
3,Mortgage,Conventional fixed mortgage,"Loan servicing, payments, escrow account",NaN,I have an open and current mortgage with Chase...,JPMorgan Chase & Co.,CA,Older American,Web
4,Mortgage,Conventional fixed mortgage,Credit decision / Underwriting,NaN,XXXX was submitted XX/XX/XXXX. At the time I s...,Rushmore Loan Management Services LLC,CA,Older American,Web


# <u> <strong> Text Preprocessing </strong> </u>

Before Machine Learning can be utilized, textual data requires preprocessing to avoid skewing analysis. Sources of such skewness and solutions to said sources are discussed below.

## <u> Stopwords </u>

A stopword is a commonly used word (such as “the”, “a”, “an”, “in”). For example, finding out that "a" is very common in my dataset is meaningless and thus, the text should be cleaned of stopwords to prevent wasting computaional power on useless analysis.

## <u> Lemmatisation </u>
Lemmatisation in linguistics is the process of grouping together the inflected forms of a word so they can be analysed as a single item, identified by the word's lemma, or dictionary form. For example, instances of "running" and "ran" in the dataset will be transformed into "run". This saves the machine algorithm from unnecessary variations.

## <u> Stemming </u>
Stemming is a text normalization technique in Natural Language Processing (NLP) that chops off the ends of words using heuristics. It uses automated rules (like "remove -ing", "remove -ed", or "convert -ies to -i") without caring for grammar or context. 

Stemming is faster than lemmatisation since only basic rules need to be followed when transforming the data instead of needing to follow the rules of the dictionary. However, stemming is not as effective in grouping related groups of words compared to lemmatisation. For example, feet and foot may be grouped seperately by stemming unlike lemmatisation.





In [22]:

stop_words_file = r"C:\Users\Lenovo\Downloads\SmartStoplist.txt" # Source: https://github.com/aneesha/RAKE/blob/master/SmartStoplist.txt

stop_words = []

with open(stop_words_file, "r") as f:
    for line in f:
        stop_words.extend(line.split()) 

# sensitive information (like names, account numbers, and dates) in public complaint data have been masked so it has to be filtered out
stop_words.extend(['xx', 'xxxx'])

def preprocess(raw_text):
    
    #regular expression keeping only letters
    letters_only_text = re.sub("[^a-zA-Z]", " ", raw_text)

    # convert to lower case and split into words 
    words = letters_only_text.lower().split()

    cleaned_words = []
    stemmer = PorterStemmer()
    
    # remove stopwords
    for word in words:
        if word not in stop_words:
            cleaned_words.append(word)
    
    # stemming words
    stemmed_words = []
    for word in cleaned_words:
        word = stemmer.stem(word)
        stemmed_words.append(word)
    
    # converting list back to string
    return " ".join(stemmed_words)



In [23]:
text_data['processed_consumer_complaint_narrative'] = text_data['consumer_complaint_narrative'].apply(preprocess) 

In [24]:
#let's check the most common words in the UP dataframe
from collections import Counter
Counter(" ".join(text_data['processed_consumer_complaint_narrative']).split()).most_common(10)

[('credit', 89173),
 ('account', 86464),
 ('call', 72017),
 ('payment', 69239),
 ('report', 60635),
 ('loan', 55512),
 ('bank', 47376),
 ('receiv', 43747),
 ('time', 43688),
 ('inform', 41166)]

# <u> <strong> Most Common Word Analysis </strong> </u>

This is the most basic form of textual analysis, seeing which words pop up the most often and gleaning insight from that. The prevalence of "payment" and "loan" in customer complaints suggest that many of the issues customers have with a bank come from the employees and systems put in place to handle loans/payments, hence warranting greater scrutiny in those areas.

This alone is too barebones to make a comprehensive analysis of the complaints so for the next step would be to see which two <strong> consecutive </strong> words are the most common.

In [31]:
from nltk.util import ngrams


n_gram = 2
n_gram_counter = Counter()

# By looping over each row, the text from one customer's complaint does not spill over into another customer's complaint
for narrative in text_data['processed_consumer_complaint_narrative']:
    n_gram_counter.update(ngrams(narrative.split(), n_gram))

top_10 = n_gram_counter.most_common(10)

for pair, count in top_10:
    print(pair, count)



('credit', 'report') 27954
('credit', 'card') 14577
('well', 'fargo') 8838
('bank', 'america') 7100
('credit', 'bureau') 6426
('collect', 'agenc') 6117
('make', 'payment') 6067
('custom', 'servic') 5734
('phone', 'call') 5644
('call', 'back') 5403


# <u> <strong> Most Common Bigram Analysis </strong> </u>

Looking at the results, it seems that bank activities that affect a customer's credit tend to attract the most complaints, indicating that improvement in this area might be necessary. Howver, before making sweeping conclusions, it's best to check the complaints that contain these common phrases to see if our understanding aligns with the actual content. 

An example of a potential mismatch would be negation. For example, the phrase "good service" may appear often in reviews of a business but actually reviewing the reviews may show that "no" appears right before "good service" in most reviews. Hence, the actual meaning of the frequent appearance of "good service" is reversed completely.

Looking at some of the reviews below, there doesn't seem to be any issues with my analysis for the frequent appearance of "credit report" in the complaints.

In [40]:
pd.set_option('display.max_colwidth', None)
mask = (text_data['consumer_complaint_narrative'].str.contains("credit", case=False, na=False) &
        text_data['consumer_complaint_narrative'].str.contains("report", case=False, na=False))
text_data[mask]['consumer_complaint_narrative'].head(5)

1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

# <u> <strong> Term Frequency-Inverse Document Frequency (TD-IDF) </strong> </u>

TD-IDF is a statistical metric used in natural language processing (NLP) to evaluate how important a word is to a document within a larger collection.
It multiplies the word's frequency in a specific text by its rarity across the entire dataset. Unlike the previous most common word/bigram analysis,
By balancing frequency with rarity, TD-IDF can filter out common words that may not have much useful meaning which the most common word/bigram analysis will not do. 

To better leverage the strengths of TD-IDF, I grouped the complaints based on the product the customers were complaining about (Debt Collection, Credit Card etc.)


In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=lambda x: x,
    token_pattern=None,
    max_df=0.5,   # optional: drop overly common words
    min_df=5      # optional: drop overly rare words
)
tfidf_matrix = vectorizer.fit_transform(text_data['processed_consumer_complaint_narrative'])
words = vectorizer.get_feature_names_out()

products = text_data['product'].unique()

for prod in products:
    mask = text_data['product'] == prod
    mean_tfidf = tfidf_matrix[mask.values].mean(axis=0).A1
    top_indices = mean_tfidf.argsort()[::-1][:10]
    top_words = [words[i] for i in top_indices]
    print(f"{prod}: {', '.join(top_words)}")

Debt collection: debt, call, collect, credit, report, compani, account, owe, number, phone
Consumer Loan: payment, car, loan, vehicl, call, credit, pay, report, account, financ
Mortgage: mortgag, loan, payment, modif, home, bank, call, time, month, ocwen
Credit card: card, credit, account, charg, payment, bank, call, balanc, fee, interest
Credit reporting: report, credit, account, equifax, experian, remov, disput, inform, transunion, file
Student loan: loan, payment, student, navient, privat, pay, school, call, month, make
Bank account or service: bank, account, check, fee, deposit, charg, money, overdraft, card, fund
Payday loan: loan, payday, call, pay, payment, cash, compani, account, paid, charg
Money transfers: money, paypal, transfer, western, account, bank, union, moneygram, wire, transact
Other financial service: check, money, fee, cash, account, servic, order, bank, told, refund
Prepaid card: card, rushcard, money, rush, deposit, access, account, fund, prepaid, call


# <u> <strong> Term Frequency-Inverse Document Frequency (TD-IDF) Analysis </strong> </u>

Looking at the results, one striking observation are the names of specific companies/programs. Examples include equifax, ocwen and rushcard (reloadable prepaid Visa was designed for unbanked individuals). The frequency in which these terms appear suggest that these companies/programs have significant issues that should be looked into.


# <u> <strong> Latent Dirichlet Allocation </strong> </u>

While most common word/bigram analysis is sufficient for exploratory analysis, a more rigorous method would be the Latent Dirichlet Allocation model. This model is a statistical approach to topic modeling, allowing for salient topics to be gleaned and classified from textual data.


In [43]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


count_vectorizer = CountVectorizer(
    tokenizer=str.split,   # since my text is already cleaned and space-joined
    preprocessor=lambda x: x,  # skip sklearn's own lowercasing/cleaning
    token_pattern=None,
    max_df=0.5,   # ignore words appearing in more than 50% of documents
    min_df=5      # ignore words appearing in fewer than 5 documents (rare/noise)
)
count_matrix = count_vectorizer.fit_transform(text_data['processed_consumer_complaint_narrative'])

lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
doc_topic_distribution = lda_model.fit_transform(count_matrix)

words = count_vectorizer.get_feature_names_out()


for topic_idx, topic in enumerate(lda_model.components_):
    print(f"Topic #{topic_idx + 1}:")
    top_words_idx = topic.argsort()[:-11:-1]  # To find top 10 words 
    top_words = [words[i] for i in top_words_idx]
    print(", ".join(top_words))

Topic #1:
call, phone, number, debt, told, compani, time, work, contact, collect
Topic #2:
report, credit, debt, account, collect, disput, remov, inform, agenc, equifax
Topic #3:
account, card, bank, charg, credit, payment, fee, check, call, money
Topic #4:
loan, mortgag, payment, home, modif, bank, month, time, properti, year
Topic #5:
loan, inquiri, student, credit, payment, navient, pay, report, school, hard


# <u> <strong> Latent Dirichlet Allocation Analysis </strong> </u>

Looking at the five topics generated by the model, it's clear that mortgages and student loans/payments, topics 4 and 5 respectively, have attracted an outsized number of complaints so it is prudent to check on the departments responsible for those two areas to search for areas of improvement.

Topics 1 and 2 indicate that debt collection by banks also result in many complaints. There may not be as many actionable steps in this situation, as debt collection is inherently combative and unpleasant but it may still be a good area for improvement.